# 北海道紋別市・湧別町 土地利用分類 2000年版

Landsat 5 TM Collection 2 SR を使用した分類

## 概要
- 2000年は直接的な教師データがないため、以下の2つの方法を提供
- **方法A**: 2020年モデルの転用（共通バンドのみ使用）
- **方法B**: 擬似教師データによる分類

## ⚠️ 精度の限界
- 2000年は実測データがないため、精度に本質的な限界がある
- 結果は「参考値」として扱い、変化検出等の相対的な比較に使用することを推奨

---
# 1. 環境セットアップ

In [ ]:
# 必要なパッケージをインストール
!pip install -q earthengine-api geopandas folium geemap

In [ ]:
# ライブラリのインポート
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import json

## 1.1 Google Earth Engine 認証・初期化

⚠️ **重要**: `YOUR_PROJECT_ID` を自分のプロジェクトIDに置き換えてください

In [ ]:
# ============================================
# ★★★ ここにプロジェクトIDを入力 ★★★
# ============================================
PROJECT_ID = 'YOUR_PROJECT_ID'  # 例: 'ee-username'

# 認証（ブラウザが開きます）
ee.Authenticate()

# 初期化
ee.Initialize(project=PROJECT_ID)

print('✅ Google Earth Engine 初期化成功！')

---
# 2. 設定パラメータ

In [ ]:
# 対象地域の設定
STUDY_AREA = {
    'min_lon': 143.0,
    'max_lon': 144.5,
    'min_lat': 43.8,
    'max_lat': 44.6
}

# 分類カテゴリ
LAND_USE_CLASSES = {
    1: 'grassland',           # 牧草地
    2: 'corn_field',          # 飼料用トウモロコシ畑
    3: 'other_cropland',      # 畑作農地
    4: 'forest',              # 森林
    5: 'urban',               # 市街地
    6: 'water',               # 水域
    7: 'bare_land'            # 裸地
}

# 分析対象年
TARGET_YEAR = 2000
START_DATE = f'{TARGET_YEAR}-05-01'
END_DATE = f'{TARGET_YEAR}-09-30'

# 雲量閾値
CLOUD_COVER_MAX = 20

# Random Forest パラメータ
RF_PARAMS = {
    'numberOfTrees': 100,
    'minLeafPopulation': 5,
    'bagFraction': 0.7,
    'seed': 42
}

print(f'対象年: {TARGET_YEAR}')

In [ ]:
# 対象地域のジオメトリ
geometry = ee.Geometry.Rectangle([
    STUDY_AREA['min_lon'], STUDY_AREA['min_lat'],
    STUDY_AREA['max_lon'], STUDY_AREA['max_lat']
])

# 地図で確認
Map = geemap.Map(center=[44.2, 143.7], zoom=9)
Map.addLayer(geometry, {'color': 'blue'}, '対象地域')
Map

---
# 3. Landsat 5 TM vs Landsat 8 OLI バンド対応

| 波長帯 | Landsat 5 TM | Landsat 8 OLI |
|-------|-------------|---------------|
| Blue | SR_B1 | SR_B2 |
| Green | SR_B2 | SR_B3 |
| Red | SR_B3 | SR_B4 |
| NIR | SR_B4 | SR_B5 |
| SWIR1 | SR_B5 | SR_B6 |
| SWIR2 | SR_B7 | SR_B7 |

---
# 4. Landsat 5 データ取得・前処理

In [ ]:
def mask_landsat5_clouds(image):
    """Landsat 5 の雲マスキング"""
    qa = image.select('QA_PIXEL')
    cloud_bit_mask = 1 << 3
    cloud_shadow_bit_mask = 1 << 4
    
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0) \
        .And(qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0))
    
    return image.updateMask(mask)


def apply_scale_factors_l5(image):
    """Landsat 5 のスケールファクター適用"""
    optical_bands = image.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']) \
        .multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, overwrite=True)


# Landsat 5 コレクションを取得
collection = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2') \
    .filterBounds(geometry) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_COVER_MAX)) \
    .map(mask_landsat5_clouds) \
    .map(apply_scale_factors_l5)

image_count = collection.size().getInfo()
print(f'✅ 取得画像数: {image_count}')

In [ ]:
# 取得した画像を表示
sample_image = collection.median()

Map2 = geemap.Map(center=[44.2, 143.7], zoom=10)
vis_tc = {'bands': ['SR_B3', 'SR_B2', 'SR_B1'], 'min': 0, 'max': 0.3}
Map2.addLayer(sample_image.clip(geometry), vis_tc, 'Landsat 5 True Color (2000)')
Map2

---
# 5. 植生指数の計算（Landsat 5用）

In [ ]:
def add_vegetation_indices_l5(image):
    """
    Landsat 5 TM 用の植生指数計算
    
    バンド対応:
    - Blue: SR_B1
    - Green: SR_B2
    - Red: SR_B3
    - NIR: SR_B4
    - SWIR1: SR_B5
    - SWIR2: SR_B7
    """
    # NDVI
    ndvi = image.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    
    # EVI
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('SR_B4'),
            'RED': image.select('SR_B3'),
            'BLUE': image.select('SR_B1')
        }
    ).rename('EVI')
    
    # NDWI
    ndwi = image.normalizedDifference(['SR_B2', 'SR_B4']).rename('NDWI')
    
    # LSWI
    lswi = image.normalizedDifference(['SR_B4', 'SR_B5']).rename('LSWI')
    
    # NDBI
    ndbi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDBI')
    
    return image.addBands([ndvi, evi, ndwi, lswi, ndbi])

print('✅ 植生指数計算関数を定義')

---
# 6. 特徴量スタックの作成

In [ ]:
def create_monthly_composites_l5(collection, year):
    """月別コンポジット作成"""
    months = [5, 6, 7, 8, 9]
    composites = []
    
    for month in months:
        start = f'{year}-{month:02d}-01'
        end = f'{year}-{month:02d}-30' if month == 9 else f'{year}-{month+1:02d}-01'
        
        monthly_col = collection.filterDate(start, end)
        composite = monthly_col.median().set('month', month)
        composite = add_vegetation_indices_l5(composite)
        
        bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7',
                 'NDVI', 'EVI', 'NDWI', 'LSWI', 'NDBI']
        renamed_bands = [f'M{month}_{b}' for b in bands]
        composite = composite.select(bands).rename(renamed_bands)
        composites.append(composite)
    
    stacked = composites[0]
    for comp in composites[1:]:
        stacked = stacked.addBands(comp)
    
    return stacked


def calculate_temporal_statistics_l5(collection):
    """時系列統計量"""
    collection_with_indices = collection.map(add_vegetation_indices_l5)
    
    indices = ['NDVI', 'EVI', 'NDWI', 'LSWI']
    stats_image = ee.Image()
    
    for idx in indices:
        idx_collection = collection_with_indices.select(idx)
        
        max_val = idx_collection.max().rename(f'{idx}_max')
        min_val = idx_collection.min().rename(f'{idx}_min')
        mean_val = idx_collection.mean().rename(f'{idx}_mean')
        std_val = idx_collection.reduce(ee.Reducer.stdDev()).rename(f'{idx}_std')
        cv = std_val.divide(mean_val.abs().add(0.001)).rename(f'{idx}_cv')
        
        stats_image = stats_image.addBands([max_val, min_val, mean_val, std_val, cv])
    
    return stats_image


def calculate_phenology_features_l5(collection, year):
    """フェノロジー特徴量"""
    july_col = collection.filterDate(f'{year}-07-01', f'{year}-07-31').map(add_vegetation_indices_l5)
    aug_col = collection.filterDate(f'{year}-08-01', f'{year}-08-31').map(add_vegetation_indices_l5)
    sept_col = collection.filterDate(f'{year}-09-01', f'{year}-09-30').map(add_vegetation_indices_l5)
    
    july_ndvi = july_col.select('NDVI').median().rename('NDVI_july')
    aug_ndvi = aug_col.select('NDVI').median().rename('NDVI_aug')
    sept_ndvi = sept_col.select('NDVI').median().rename('NDVI_sept')
    
    ndvi_diff_aug_july = aug_ndvi.subtract(july_ndvi).rename('NDVI_diff_aug_july')
    ndvi_diff_sept_aug = sept_ndvi.subtract(aug_ndvi).rename('NDVI_diff_sept_aug')
    summer_max_ndvi = july_col.merge(aug_col).select('NDVI').max().rename('NDVI_summer_max')
    
    phenology = ee.Image.cat([
        july_ndvi, aug_ndvi, sept_ndvi,
        ndvi_diff_aug_july, ndvi_diff_sept_aug,
        summer_max_ndvi
    ])
    
    return phenology


def add_terrain_features():
    """地形特徴量"""
    dem = ee.Image('USGS/SRTMGL1_003')
    
    elevation = dem.select('elevation').rename('elevation')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    aspect_sin = aspect.multiply(3.14159 / 180).sin().rename('aspect_sin')
    aspect_cos = aspect.multiply(3.14159 / 180).cos().rename('aspect_cos')
    
    return ee.Image.cat([elevation, slope, aspect_sin, aspect_cos])

print('✅ 特徴量計算関数を定義')

In [ ]:
# 特徴量スタック作成
print('特徴量スタック作成中...')

monthly_composites = create_monthly_composites_l5(collection, TARGET_YEAR)
print('  - 月別コンポジット完了')

temporal_stats = calculate_temporal_statistics_l5(collection)
print('  - 時系列統計量完了')

phenology = calculate_phenology_features_l5(collection, TARGET_YEAR)
print('  - フェノロジー特徴量完了')

terrain = add_terrain_features()
print('  - 地形データ完了')

feature_stack = ee.Image.cat([
    monthly_composites,
    temporal_stats,
    phenology,
    terrain
]).clip(geometry)

band_names = feature_stack.bandNames().getInfo()
print(f'\n✅ 特徴量スタック作成完了')
print(f'特徴量数: {len(band_names)}')

---
# 7. 方法B: 擬似教師データによる分類

## スペクトル閾値を使用した自動分類

既存の土地被覆データやスペクトル特性から擬似的な教師データを作成

In [ ]:
def create_pseudo_training_from_spectral_thresholds(feature_stack):
    """
    スペクトル閾値を使用した擬似クラス作成
    
    ⚠️ 注意: これは参考値であり、実際の土地利用とは異なる可能性がある
    """
    # 必要なバンドを取得
    ndvi_max = feature_stack.select('NDVI_max')
    ndwi_mean = feature_stack.select('NDWI_mean')
    ndvi_aug = feature_stack.select('NDVI_aug')
    elevation = feature_stack.select('elevation')
    
    # 閾値ベースの分類
    # 水域: NDWI > 0
    water = ndwi_mean.gt(0).multiply(6)
    
    # 森林: NDVI_max > 0.7 かつ elevation > 100m
    forest = ndvi_max.gt(0.7).And(elevation.gt(100)).multiply(4)
    
    # 市街地: NDVI_max < 0.3
    urban = ndvi_max.lt(0.3).And(ndwi_mean.lt(0)).multiply(5)
    
    # 農地（高NDVI平野部）
    cropland_mask = ndvi_max.gt(0.5).And(elevation.lt(100))
    
    # トウモロコシ: 8月NDVIが高い（> 0.75）
    corn = cropland_mask.And(ndvi_aug.gt(0.75)).multiply(2)
    
    # 牧草地: 8月NDVIが中程度（0.5-0.75）
    grassland = cropland_mask.And(ndvi_aug.gte(0.5)).And(ndvi_aug.lte(0.75)).multiply(1)
    
    # その他農地
    other_crop = cropland_mask.And(ndvi_aug.lt(0.5)).multiply(3)
    
    # 裸地
    bare = ndvi_max.lt(0.2).And(ndwi_mean.lt(0)).multiply(7)
    
    # 統合（優先度順）
    pseudo_class = water \
        .where(forest.gt(0), forest) \
        .where(urban.gt(0), urban) \
        .where(corn.gt(0), corn) \
        .where(grassland.gt(0), grassland) \
        .where(other_crop.gt(0), other_crop) \
        .where(bare.gt(0), bare)
    
    return pseudo_class.rename('class')

# 擬似分類を作成
pseudo_class = create_pseudo_training_from_spectral_thresholds(feature_stack)

print('✅ 擬似教師データ作成完了')

In [ ]:
# 擬似分類結果を確認
Map3 = geemap.Map(center=[44.2, 143.7], zoom=10)

class_palette = [
    '#90EE90',  # 1: 牧草地
    '#FFD700',  # 2: トウモロコシ
    '#FFA500',  # 3: 畑作
    '#006400',  # 4: 森林
    '#FF0000',  # 5: 市街地
    '#0000FF',  # 6: 水域
    '#808080',  # 7: 裸地
]

vis_class = {'min': 1, 'max': 7, 'palette': class_palette}
Map3.addLayer(pseudo_class.clip(geometry), vis_class, '擬似分類（閾値ベース）')

legend_dict = {
    '1: 牧草地': '#90EE90',
    '2: トウモロコシ畑': '#FFD700',
    '3: 畑作農地': '#FFA500',
    '4: 森林': '#006400',
    '5: 市街地': '#FF0000',
    '6: 水域': '#0000FF',
    '7: 裸地': '#808080',
}
Map3.add_legend(title='土地利用', legend_dict=legend_dict)

Map3

In [ ]:
# 擬似分類からサンプルポイントを抽出
def sample_pseudo_training_points(pseudo_class_image, geometry, samples_per_class=300):
    """擬似分類画像からストラタファイドサンプリング"""
    samples = pseudo_class_image.stratifiedSample(
        numPoints=samples_per_class,
        classBand='class',
        region=geometry,
        scale=30,
        seed=42,
        geometries=True
    )
    return samples

training_samples = sample_pseudo_training_points(pseudo_class, geometry)
print(f'✅ サンプル抽出完了: {training_samples.size().getInfo()} ポイント')

---
# 8. Random Forest 分類

In [ ]:
# 教師サンプルにバンド値を抽出
training_with_features = feature_stack.sampleRegions(
    collection=training_samples,
    properties=['class'],
    scale=30,
    tileScale=8
)

print(f'訓練サンプル数: {training_with_features.size().getInfo()}')

In [ ]:
# Random Forest 分類器を訓練
classifier = ee.Classifier.smileRandomForest(**RF_PARAMS).train(
    features=training_with_features,
    classProperty='class',
    inputProperties=band_names
)

print('✅ Random Forest 訓練完了')

In [ ]:
# 分類実行
classified = feature_stack.classify(classifier)

print('✅ 分類完了')

In [ ]:
# 分類結果を表示
Map4 = geemap.Map(center=[44.2, 143.7], zoom=10)

Map4.addLayer(classified.clip(geometry), vis_class, '土地利用分類 2000年')
Map4.add_legend(title='土地利用 2000', legend_dict=legend_dict)

Map4

---
# 9. 2020年との比較（オプション）

2020年の分類結果と並べて表示

In [ ]:
# 2000年と2020年の比較マップ
# ※ 2020年の分類結果がある場合に使用

Map5 = geemap.Map(center=[44.2, 143.7], zoom=10)

# 2000年の分類結果
Map5.addLayer(classified.clip(geometry), vis_class, '2000年 土地利用')

# 2020年の分類結果（別途読み込み）
# classified_2020 = ee.Image('users/YOUR_USERNAME/landuse_2020')
# Map5.addLayer(classified_2020.clip(geometry), vis_class, '2020年 土地利用')

Map5.add_legend(title='土地利用', legend_dict=legend_dict)
Map5

---
# 10. 結果のエクスポート

In [ ]:
# Google Drive にエクスポート
export_task = ee.batch.Export.image.toDrive(
    image=classified.toInt8(),
    description='Mombetsu_LandUse_2000_MethodB',
    folder='LandUse_Mombetsu',
    region=geometry,
    scale=30,
    maxPixels=1e13,
    crs='EPSG:32654'
)

export_task.start()

print('✅ エクスポートタスク開始')
print(f'タスクID: {export_task.id}')
print('\nGoogle Drive の "LandUse_Mombetsu" フォルダに保存されます')

In [ ]:
# タスク状態確認
import time

print('エクスポート状態を確認中...')
while True:
    status = export_task.status()
    state = status['state']
    print(f'状態: {state}')
    
    if state in ['COMPLETED', 'FAILED', 'CANCELLED']:
        break
    
    time.sleep(30)

if state == 'COMPLETED':
    print('\n✅ エクスポート完了！')
else:
    print(f'\n❌ エクスポート失敗: {status}')

---
# 11. まとめと精度の限界

## ⚠️ 精度の限界

**2000年の分類には以下の限界があります:**

1. **直接的な教師データがない**
   - 擬似教師データはスペクトル閾値に基づくため、実際の土地利用と異なる可能性

2. **牧草地とトウモロコシの識別精度**
   - 2000年当時の栽培パターンが現在と異なる可能性
   - スペクトル閾値は現在の知見に基づいており、2000年に最適化されていない

3. **土地利用変化の影響**
   - 20年間で農地転用、森林伐採、都市化などの変化がある

## 推奨される使用方法

- **変化検出**: 2020年との相対的な比較に使用
- **傾向把握**: 大まかな土地利用パターンの把握
- **参考値**: 精度検証なしの参考データとして扱う

## 精度向上の可能性

1. 2000年頃の航空写真や地図との照合
2. 農林業センサス（2000年）との統合
3. 国土数値情報の過去データとの照合